# dgm-eval on the Traffic Signs 200k sample sets (nine runs, one notebook)

Same install, mount and environment cells as the Step10 / Dec 2025 PottedPlant notebook; only the dataset cells and the run cell changed.
The run cell loops over **every `*_n3000_*.zip` in one Drive folder**, stages each as a flat folder, scores it with DINOv2 and
Inception in the documented argument order, adds the failure rate and the global colour statistics, and appends one row per set to
an Excel sheet on Drive after every set, so a disconnect loses nothing.

**Run order:** cells 1–2, then the package-pins cell → **Runtime ▸ Restart session** → continue from the version-check cell.

**Before you start:** copy the nine zips from `samples_Step11` into `MyDrive/PhD/Cityscapes/Sets_200k/` (or change `SETS_DIR` in CONFIG).

dgm-eval argument order (README and `__main__.py`): `python -m dgm_eval <REAL_TRAIN> <GENERATED> --test_path <REAL_TEST>`.


In [ ]:
%cd /content
!rm -rf dgm-eval
!git clone https://github.com/layer6ai-labs/dgm-eval.git
%cd dgm-eval


In [ ]:
!pip install --quiet --no-deps -e .


**Restart Session after the cell below** (Runtime ▸ Restart session), then continue from the version-check cell.


In [ ]:
# Core scientific stack (plays well on Colab Py3.12)
!pip install -q "numpy==2.2.6" "scipy==1.16.1" "pandas==2.2.2" "scikit-learn==1.7.2"

# Vision + image utils
!pip install -q "opencv-python==4.12.0.88" "pillow==10.4.0" "scikit-image==0.24.0"

# Encoders used by dgm-eval  (xformers deliberately NOT installed any more: the Oct 2026 Colab image ships
# PyTorch 2.11 / Python 3.13, the xformers wheel was built for PyTorch 2.10 / Python 3.10, and DINOv2 then crashes
# inside memory_efficient_attention. DINOv2 runs its plain attention instead; the features are the same.)
!pip install -q "open_clip_torch==2.19.0" "timm==0.9.16" "transformers==4.45.2"


In [ ]:
import numpy, pandas, sklearn, scipy, PIL, cv2, skimage
import torch, torchvision
import open_clip, timm, transformers

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("SciPy:", scipy.__version__)
print("Pillow:", PIL.__version__)
print("cv2:", cv2.__version__)
print("skimage:", skimage.__version__)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("open_clip:", getattr(open_clip, "__version__", "ok"))
print("timm:", timm.__version__)
print("transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")
print("All good.")


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# create symbolic link
!ln -s /content/gdrive/My\ Drive/ /mydrive

# list gdrive content
!ls /mydrive


## Real sets

Training set = the Traffic Signs 256 px images the models were trained on. Held-out test set = `TestSmall.zip`, the
set the November 2025 table runs used. `TEST_SRC` may be a zip or a folder on Drive.

In [ ]:
REAL_ZIP = "/content/gdrive/MyDrive/PhD/TrafficSigns/TrafficSigns_img.zip"       # <-- 3,009 training images
TEST_SRC = "/content/gdrive/MyDrive/PhD/TrafficSigns/TestSmall.zip"               # <-- folder, or a .zip of it

import os, shutil, zipfile, glob
REAL_DIR = "/content/dgm-eval/TrafficSigns_img"
TEST_DIR = "/content/dgm-eval/TrafficSigns_test"

def flatten_into(src_root, dst):
    os.makedirs(dst, exist_ok=True); n = 0
    for dp, dn, fn in os.walk(src_root):
        if "__MACOSX" in dp:
            continue
        for f in fn:
            if f.lower().endswith((".png", ".jpg", ".jpeg")) and not f.startswith("._"):
                shutil.copy2(os.path.join(dp, f), os.path.join(dst, f)); n += 1
    return n

if not (os.path.isdir(REAL_DIR) and os.listdir(REAL_DIR)):
    tmp = "/content/dgm-eval/_real_raw"; os.makedirs(tmp, exist_ok=True)
    with zipfile.ZipFile(REAL_ZIP) as z:
        z.extractall(tmp)
    print("real train images:", flatten_into(tmp, REAL_DIR)); shutil.rmtree(tmp)
if not (os.path.isdir(TEST_DIR) and os.listdir(TEST_DIR)):
    if TEST_SRC.lower().endswith(".zip"):
        tmp = "/content/dgm-eval/_test_raw"; os.makedirs(tmp, exist_ok=True)
        with zipfile.ZipFile(TEST_SRC) as z:
            z.extractall(tmp)
        print("real test images:", flatten_into(tmp, TEST_DIR)); shutil.rmtree(tmp)
    else:
        print("real test images:", flatten_into(TEST_SRC, TEST_DIR))
print("real_train", len(os.listdir(REAL_DIR)), "| real_test", len(os.listdir(TEST_DIR)))


In [ ]:
%cd /content/dgm-eval


In [ ]:
# ==== keep TensorFlow out of the way ====================================================================
# open_clip_torch 2.19.0 pins protobuf<4, and Colab's TensorFlow (Python 3.13 image) needs protobuf>=5.26.
# `transformers` imports TensorFlow whenever it sees it installed, which is where the
# "cannot import name 'runtime_version' from 'google.protobuf'" crash came from. dgm-eval never uses TensorFlow,
# so we tell `transformers` to ignore it. The two variables are inherited by every !python command below.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# XFORMERS_DISABLED=1 is read by the DINOv2 hub code (dinov2/layers/attention.py, swiglu_ffn.py, block.py):
# with it set, DINOv2 uses its ordinary PyTorch attention instead of xformers' memory_efficient_attention,
# which cannot run when the installed xformers wheel does not match Colab's PyTorch build.
os.environ["XFORMERS_DISABLED"] = "1"

# smoke test: this must print the encoder list (inception, dinov2, ...) and no traceback
!USE_TF=0 USE_TORCH=1 XFORMERS_DISABLED=1 python -c "import google.protobuf as p; print('protobuf', p.__version__); from dgm_eval.models import MODELS; print('dgm-eval import OK:', list(MODELS))"


In [ ]:
# ==== FALLBACKS - run ONLY if the smoke test above still shows a traceback ===============================
# F1: newer protobuf (overrides open_clip's old pin; open_clip only needs protobuf for text tokenisers, not images)
#!pip install -q "protobuf>=5.29,<7"
# F2: drop the unused data2vec encoder, which is the only thing in dgm-eval that imports `transformers`
#!sed -i 's/^from .data2vec import HuggingFaceTransformerEncoder/HuggingFaceTransformerEncoder = None  # disabled/' /content/dgm-eval/dgm_eval/models/__init__.py /content/dgm-eval/dgm_eval/models/load_encoder.py
# F3: if DINOv2 still dies inside xformers "memory_efficient_attention" (an old xformers wheel is installed in this
#     session), remove xformers altogether; DINOv2 then falls back to plain attention by itself
#!pip uninstall -y -q xformers
# then re-run the smoke test cell above


## CONFIG and the loop over all sets

Each zip is staged as one flat folder of sample PNGs (the sampler's `meta.json` is read for the boxes and then left out), checked for
corrupt files, scored by DINOv2 (run A) and Inception (run B), measured for the failure rate and the colour statistics, and written
as one row of `Traffic_200k_metrics.xlsx`. Sets already present in the sheet are skipped, so the cell can be re-run after a disconnect.
The real-set features are cached by dgm-eval in `experiments_traffic200k` after the first set.

In [ ]:
# ==== CONFIG ===============================================================================
SETS_DIR    = "/content/gdrive/MyDrive/PhD/TrafficSigns/Sets_200k"          # <-- the nine *_n3000_*.zip files from samples_Step11
RESULTS_DIR = "/content/gdrive/MyDrive/PhD/TrafficSigns/Sets_200k_metrics"  # txt per run + the xlsx
NSAMPLE     = 3000
# ============================================================================================
import os, re, glob, json, shutil, zipfile, subprocess, time
import numpy as np, pandas as pd
from PIL import Image
os.makedirs(RESULTS_DIR, exist_ok=True)
XLSX = f"{RESULTS_DIR}/Traffic_200k_metrics.xlsx"
ENV = dict(os.environ, USE_TF="0", USE_TORCH="1", XFORMERS_DISABLED="1")
IMG_EXT = (".png", ".jpg", ".jpeg")

def lap_var(g):
    k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32); H, W = g.shape; o = np.zeros((H - 2, W - 2), np.float32)
    for dy in range(3):
        for dx in range(3):
            o += k[dy, dx] * g[dy:dy + H - 2, dx:dx + W - 2]
    return float(o.var())

def image_stats(folder):
    files = sorted(f for f in os.listdir(folder) if f.lower().endswith(IMG_EXT))
    br, tint, sat, con, grain = [], [], [], [], []
    for f in files:
        im = Image.open(os.path.join(folder, f)).convert("RGB"); a = np.asarray(im, dtype=np.float32)
        br.append(a.mean()); con.append(a.std()); grain.append(lap_var(np.asarray(im.convert("L"), dtype=np.float32)))
        s = im.resize((32, 32), Image.BOX); m = np.asarray(s, dtype=np.float32).reshape(-1, 3).mean(0)
        tint.append(np.std(m)); sat.append(np.asarray(s.convert("HSV"), dtype=np.float32)[..., 1].mean())
    g = np.array(grain)
    return dict(n_images=len(files), brightness=float(np.mean(br)), tint=float(np.mean(tint)), saturation=float(np.mean(sat)),
                contrast=float(np.mean(con)), grain_median=float(np.median(g)), failure_rate_pct=float(100.0 * (g > 2000).mean()))

def parse(txt_path):
    out = {}
    with open(txt_path) as f:
        for line in f:
            m = re.match(r'^([A-Za-z_\-]+):\s*([-+0-9.eE]+)$', line.strip())
            if m:
                try:
                    out[m.group(1)] = float(m.group(2))
                except ValueError:
                    pass
    return out

def run_dgm(model, metrics, gen_dir, tag):
    txt = f"{RESULTS_DIR}/{tag}_{model}.txt"
    cmd = ["python", "-m", "dgm_eval", REAL_DIR, gen_dir, "--test_path", TEST_DIR, "--model", model, "--metrics", *metrics.split(),
           "--device", "cuda", "--batch_size", "128", "--nsample", str(NSAMPLE), "--seed", "42",
           "--output_dir", "/content/dgm-eval/experiments_traffic200k"]
    with open(txt, "w") as f:
        p = subprocess.run(cmd, cwd="/content/dgm-eval", env=ENV, stdout=f, stderr=subprocess.STDOUT)
    res = parse(txt)
    if p.returncode != 0 or "fd" not in res:
        # dgm-eval's C_T scores fail on some sets (k-means cells without samples); drop them and retry rather than stop
        for drop in ("ct_modified", "ct"):
            if drop in metrics.split():
                print(f"   [{model}] dgm-eval failed; retrying without {drop}")
                return run_dgm(model, " ".join(m for m in metrics.split() if m != drop), gen_dir, tag)
        print(open(txt).read()[-3000:]); raise RuntimeError(f"dgm-eval failed for {tag} / {model}: see {txt}")
    return res

done = set()
if os.path.exists(XLSX):
    done = set(pd.read_excel(XLSX, index_col=0).index.astype(str))
    print("already scored:", sorted(done))

zips = sorted(glob.glob(f"{SETS_DIR}/*_n3000_*.zip"))
assert zips, f"no *_n3000_*.zip in {SETS_DIR}"
print(len(zips), "sets found")
for zp in zips:
    tag = os.path.basename(zp)[:-4]
    if tag in done:
        print("skip (done):", tag); continue
    t0 = time.time(); print("\n=====", tag, "=====")
    raw = f"/content/dgm-eval/_raw_{tag}"; gen = f"/content/dgm-eval/gen_{tag}"
    shutil.rmtree(raw, ignore_errors=True); shutil.rmtree(gen, ignore_errors=True); os.makedirs(gen)
    with zipfile.ZipFile(zp) as z:
        z.extractall(raw)
    meta = {}
    for dp, dn, fn in os.walk(raw):
        for f in fn:
            if f == "meta.json":
                meta = json.load(open(os.path.join(dp, f)))
            elif f.startswith("sample_") and f.lower().endswith(".png"):
                shutil.copy2(os.path.join(dp, f), os.path.join(gen, f))
    shutil.rmtree(raw)
    bad = 0
    for f in sorted(os.listdir(gen)):
        try:
            with Image.open(os.path.join(gen, f)) as im:
                im.verify()
        except Exception:
            os.remove(os.path.join(gen, f)); bad += 1
    stats = image_stats(gen)
    print(f"staged {stats['n_images']} images ({bad} corrupt removed) | brightness {stats['brightness']:.1f} tint {stats['tint']:.1f} "
          f"saturation {stats['saturation']:.1f} contrast {stats['contrast']:.1f} failure rate {stats['failure_rate_pct']:.2f}%")
    A = run_dgm("dinov2", "fd kd prdc fd-infinity authpct fls fls_overfit ct ct_modified", gen, tag)
    B = run_dgm("inception", "fd kd prdc fd-infinity is", gen, tag)
    row = {"dinov2_" + k: v for k, v in A.items()}
    row.update({"inception_" + k: v for k, v in B.items()})
    row.update(stats)
    cost = meta.get("cost", {})
    row.update({"unet_evals_per_image": cost.get("unet_evals_per_image"), "seconds_per_image": cost.get("seconds_per_image"), "gpu": cost.get("device")})
    args = meta.get("args", {})
    row.update({"timestep_respacing": args.get("timestep_respacing"), "mask_cfg_scale": args.get("mask_cfg_scale"),
                "roi_bump_mode": args.get("roi_bump_mode"), "color_guide_mode": args.get("color_guide_mode"), "sigma_mode": args.get("sigma_mode")})
    table = pd.read_excel(XLSX, index_col=0) if os.path.exists(XLSX) else pd.DataFrame()
    table = pd.concat([table, pd.DataFrame([row], index=[tag])])
    table.index.name = "set"; table.to_excel(XLSX)
    print(f"DINOv2 FD {A.get('fd'):.2f} | KD {A.get('kd_value', float('nan')):.4f} | FID {B.get('fd'):.2f} | "
          f"precision {A.get('precision', float('nan')):.3f} recall {A.get('recall', float('nan')):.3f} | {time.time() - t0:.0f} s -> saved {XLSX}")
    shutil.rmtree(gen, ignore_errors=True)
print("\nall sets done")


## The table

Paper columns: DINOv2 FD = `dinov2_fd`, KD = `dinov2_kd_value`, FID = `inception_fd`, Precision/Recall = `dinov2_precision` / `dinov2_recall`,
FLS-Overfit = `dinov2_fls_overfit`, C_T = `dinov2_ct_score`. The extra columns are the failure rate (share of images whose Laplacian
variance exceeds 2000, the grainy failure mode), the global colour statistics against the real values (72 / 5.0 / 46 / 35), and the
measured cost per image from the sampler.

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
t = pd.read_excel(XLSX, index_col=0)
cols = [c for c in ["dinov2_fd", "dinov2_kd_value", "inception_fd", "dinov2_precision", "dinov2_recall", "dinov2_fls_overfit", "dinov2_ct_score",
                    "failure_rate_pct", "brightness", "tint", "saturation", "contrast", "unet_evals_per_image", "seconds_per_image"] if c in t.columns]
print(t[cols].sort_values("dinov2_fd").round(3))


## Fill in the C_T columns and audit the sheet

dgm-eval's C_T code crashes on some sets (k-means cells without test samples); the scoring loop keeps such a row without `ct_modified`. This cell re-runs only `ct ct_modified` for the blank rows with other seeds and writes the values into the sheet, after comparing every FD/KD/precision/recall in the sheet with the raw dgm-eval text files. Run after the CONFIG cell, same session.


In [ ]:
# ==== FILL IN THE C_T COLUMNS + AUDIT THE SHEET AGAINST THE RAW dgm-eval FILES ==============================
# Run AFTER the CONFIG cell, in the same session (CONFIG only skips the sets already scored, so it is quick).
#  1) Audit: every FD / FD-infinity / KD / precision / recall in the xlsx is compared with the raw <set>_dinov2.txt and
#     <set>_inception.txt files written by dgm-eval; any cell that differs is listed.
#  2) Fill: for rows whose dinov2_ct_modified is blank, dgm-eval is re-run for "ct ct_modified" only, trying several seeds
#     (the k-means cells change with the seed, which is what makes dgm-eval's C_T code crash); the first seed that works is
#     written into the sheet (both ct and ct_modified from that run, plus the seed).
ROWS  = "final"          # "final" = rows of the final recipe (anc500 / anc_cfg175 / the ADM anc_lv row); "all" = every blank row
SEEDS = [42, 43, 44, 45, 46]
# ==========================================================================================================
import os, shutil, zipfile, subprocess, time
import pandas as pd
assert all(n in globals() for n in ("REAL_DIR", "TEST_DIR", "parse", "RESULTS_DIR", "XLSX", "SETS_DIR", "ENV", "NSAMPLE")), \
    "run the cells above (including CONFIG) first"
table = pd.read_excel(XLSX, index_col=0)
table.index = table.index.astype(str)

# ---- 1. audit --------------------------------------------------------------------------------------------
checks = {"dinov2": ["fd", "fd_infinity_value", "kd_value", "precision", "recall"],
          "inception": ["fd", "fd_infinity_value", "kd_value"]}
n_ok = n_bad = n_missing = 0
for tag in table.index:
    for model, keys in checks.items():
        txt = f"{RESULTS_DIR}/{tag}_{model}.txt"
        if not os.path.exists(txt):
            n_missing += 1
            print(f"[audit] no raw file for {tag} / {model}")
            continue
        raw = parse(txt)
        for k in keys:
            col = f"{model}_{k}"
            if k not in raw or col not in table.columns or pd.isna(table.loc[tag, col]):
                continue
            if abs(float(raw[k]) - float(table.loc[tag, col])) > 1e-4:
                n_bad += 1
                print(f"[audit] MISMATCH {tag} {col}: sheet {float(table.loc[tag, col]):.4f}   raw file {raw[k]:.4f}")
            else:
                n_ok += 1
print(f"[audit] {n_ok} values match the raw files, {n_bad} differ, {n_missing} raw files missing")

# ---- 2. fill C_T -----------------------------------------------------------------------------------------
def stage(tag):
    zp = f"{SETS_DIR}/{tag}.zip"
    assert os.path.exists(zp), f"zip not found: {zp}"
    raw = f"/content/dgm-eval/_raw_{tag}"
    gen = f"/content/dgm-eval/gen_{tag}"
    shutil.rmtree(raw, ignore_errors=True)
    shutil.rmtree(gen, ignore_errors=True)
    os.makedirs(gen)
    with zipfile.ZipFile(zp) as z:
        z.extractall(raw)
    for dp, dn, fn in os.walk(raw):
        for f in fn:
            if f.startswith("sample_") and f.lower().endswith(".png"):
                shutil.copy2(os.path.join(dp, f), os.path.join(gen, f))
    shutil.rmtree(raw)
    return gen

def run_ct(gen, tag, seed):
    txt = f"{RESULTS_DIR}/{tag}_dinov2_ct_seed{seed}.txt"
    cmd = ["python", "-m", "dgm_eval", REAL_DIR, gen, "--test_path", TEST_DIR, "--model", "dinov2",
           "--metrics", "ct", "ct_modified", "--device", "cuda", "--batch_size", "128", "--nsample", str(NSAMPLE),
           "--seed", str(seed), "--output_dir", "/content/dgm-eval/experiments_ct"]
    with open(txt, "w") as f:
        p = subprocess.run(cmd, cwd="/content/dgm-eval", env=ENV, stdout=f, stderr=subprocess.STDOUT)
    res = parse(txt)
    return res if (p.returncode == 0 and "ct" in res and "ct_modified" in res) else None

if "dinov2_ct_modified" not in table.columns:
    table["dinov2_ct_modified"] = float("nan")
todo = [t for t in table.index
        if pd.isna(table.loc[t, "dinov2_ct_modified"]) and (ROWS == "all" or "_anc500" in t or "anc_cfg175" in t or "anc_lv" in t)]
print(f"\n{len(todo)} rows to fill: {todo}")
for tag in todo:
    t0 = time.time()
    gen = stage(tag)
    got, used = None, None
    for s in SEEDS:
        got = run_ct(gen, tag, s)
        if got:
            used = s
            break
        print(f"   {tag}: C_T failed with seed {s}, trying the next seed")
    if got:
        table.loc[tag, "dinov2_ct"] = got["ct"]
        table.loc[tag, "dinov2_ct_modified"] = got["ct_modified"]
        table.loc[tag, "ct_seed"] = used
        table.index.name = "set"
        table.to_excel(XLSX)
        print(f"{tag}: ct {got['ct']:.3f}  ct_modified {got['ct_modified']:.3f}  (seed {used}, {time.time() - t0:.0f} s) -> saved")
    else:
        print(f"{tag}: C_T could not be computed with any seed; left blank (report as '-')")
    shutil.rmtree(gen, ignore_errors=True)
print("\ndone")


## Sampling uncertainty (AE point 8): paired bootstrap of the FD

Rows are chosen from the sheet automatically (final recipe + the ADM row; LSUN-initialised rows only when they exist). Stores the DINOv2 features with dgm-eval `--save`, draws 200 paired bootstrap replicates and reports the spread per model and the interval of the paired difference between the FDGAN recipe row and every other row. Writes `<dataset>_bootstrap_FD.xlsx` next to the metrics sheet.


In [ ]:
# ==== SAMPLING UNCERTAINTY (AE point 8): paired bootstrap of the DINOv2 FD for the final-recipe rows =======================
# Rows are taken from this notebook's sheet automatically: every row sampled with the final recipe (anc500 / anc_cfg175) plus the
# ADM anc_lv row; when LSUN-initialised rows exist (COCO) only those are used. A = the FDGAN recipe row (…anc500_cfg15).
# For each row, dgm-eval is run once more with --save so that its DINOv2 features (and the real set's) are stored; then B
# bootstrap replicates: one resample of the real features shared by all models (pairs the comparisons) and one resample of
# each generated set; FD per replicate with dgm-eval's own FD function. Bootstrap FD values are biased upward compared with the
# point estimate (resampling with replacement repeats images), so only spreads, intervals and paired differences are
# reported, never the bootstrap mean as a new FD.  Run after the CONFIG cell, same session.  ~3-4 min per row + ~1 s per replicate.
B, SEED = 200, 0
# =========================================================================================================================
import os, glob, shutil, zipfile, subprocess, time
import numpy as np, pandas as pd
from dgm_eval.metrics.fd import compute_efficient_FD_with_reps
assert all(n in globals() for n in ("REAL_DIR", "SETS_DIR", "RESULTS_DIR", "XLSX", "ENV", "NSAMPLE")), "run the cells above (incl. CONFIG) first"
WHOLE_XLSX = XLSX
sheet = pd.read_excel(WHOLE_XLSX, index_col=0)
sheet.index = sheet.index.astype(str)
ROWS = [t for t in sheet.index if ("_anc500" in t or "anc_cfg175" in t or "anc_lv" in t)]
if any("_lsun_" in t for t in ROWS):
    ROWS = [t for t in ROWS if "_lsun_" in t]
cands = [t for t in ROWS if "fdgan_mask_" in t and "smallD" not in t and "rep2" not in t and t.endswith("anc500_cfg15")]
assert cands, "no FDGAN recipe row (...anc500_cfg15) in the sheet yet"
A = cands[0]
PAIRS = [(A, r) for r in ROWS if r != A]
DATASET = os.path.basename(XLSX).split("_200k")[0]
print("rows:", ROWS, "\nreference row A:", A)
CACHE = "/content/dgm-eval/reps_cache"
os.makedirs(CACHE, exist_ok=True)
REAL_NAME = os.path.basename(os.path.normpath(REAL_DIR))

def stage(tag):
    zp = f"{SETS_DIR}/{tag}.zip"
    assert os.path.exists(zp), f"zip not found: {zp}"
    raw, gen = f"/content/dgm-eval/_raw_{tag}", f"/content/dgm-eval/gen_{tag}"
    shutil.rmtree(raw, ignore_errors=True); shutil.rmtree(gen, ignore_errors=True); os.makedirs(gen)
    with zipfile.ZipFile(zp) as z:
        z.extractall(raw)
    for dp, dn, fn in os.walk(raw):
        for f in fn:
            if f.startswith("sample_") and f.lower().endswith(".png"):
                shutil.copy2(os.path.join(dp, f), os.path.join(gen, f))
    shutil.rmtree(raw)
    return gen

def find_reps(name_part):
    hits = [f for f in glob.glob(f"{CACHE}/reps_*.npz") if os.path.basename(f).startswith(f"reps_{name_part}_dinov2_")]
    return hits[0] if len(hits) == 1 else None

def features(tag):
    if find_reps(f"gen_{tag}") is None or find_reps(REAL_NAME) is None:
        gen = stage(tag)
        cmd = ["python", "-m", "dgm_eval", REAL_DIR, gen, "--model", "dinov2", "--metrics", "fd", "--device", "cuda", "--batch_size", "128",
               "--nsample", str(NSAMPLE), "--seed", "42", "--save", "--output_dir", CACHE]
        with open(f"{RESULTS_DIR}/{tag}_dinov2_features_for_bootstrap.txt", "w") as f:
            p = subprocess.run(cmd, cwd="/content/dgm-eval", env=ENV, stdout=f, stderr=subprocess.STDOUT)
        shutil.rmtree(gen, ignore_errors=True)
        if p.returncode != 0 or find_reps(f"gen_{tag}") is None:
            print(open(f"{RESULTS_DIR}/{tag}_dinov2_features_for_bootstrap.txt").read()[-2000:])
            print("files in cache:", sorted(os.listdir(CACHE)))
            raise RuntimeError(f"feature extraction failed for {tag}")
    real = np.load(find_reps(REAL_NAME), allow_pickle=True)["reps"].astype(np.float64)
    gen = np.load(find_reps(f"gen_{tag}"), allow_pickle=True)["reps"].astype(np.float64)
    return real, gen

t0 = time.time()
feats = {}
for tag in ROWS:
    real, feats[tag] = features(tag)
    print(f"features ready: {tag}  gen {feats[tag].shape}  real {real.shape}  ({time.time() - t0:.0f} s)")

rng = np.random.default_rng(SEED)
n_real = len(real)
boot = {tag: np.zeros(B) for tag in ROWS}
for b in range(B):
    ir = rng.integers(0, n_real, n_real)
    rb = real[ir]
    for tag in ROWS:
        g = feats[tag]
        boot[tag][b] = compute_efficient_FD_with_reps(rb, g[rng.integers(0, len(g), len(g))])
    if (b + 1) % 25 == 0:
        print(f"bootstrap {b + 1}/{B}  ({time.time() - t0:.0f} s)")

try:
    point = pd.read_excel(WHOLE_XLSX, index_col=0)["dinov2_fd"].astype(float)
except Exception as e:
    print("point FD not available:", e)
    point = pd.Series(dtype=float)
per_model = pd.DataFrame({
    "fd_point": [point.get(t, np.nan) for t in ROWS],
    "boot_mean": [boot[t].mean() for t in ROWS],
    "boot_std": [boot[t].std(ddof=1) for t in ROWS],
    "ci95_low": [np.percentile(boot[t], 2.5) for t in ROWS],
    "ci95_high": [np.percentile(boot[t], 97.5) for t in ROWS],
    "n_gen": [len(feats[t]) for t in ROWS], "n_real": n_real, "B": B}, index=ROWS)
pairs = pd.DataFrame([{
    "A": a, "B": b2, "diff_point": point.get(a, np.nan) - point.get(b2, np.nan),
    "diff_boot_mean": (boot[a] - boot[b2]).mean(), "diff_ci95_low": np.percentile(boot[a] - boot[b2], 2.5),
    "diff_ci95_high": np.percentile(boot[a] - boot[b2], 97.5), "P(A better)": float((boot[a] < boot[b2]).mean())} for a, b2 in PAIRS])
pd.DataFrame(boot).to_csv(f"{RESULTS_DIR}/{DATASET}_bootstrap_replicates.csv", index_label="replicate")   # raw FD per replicate and row: any pair can be compared later
out = f"{RESULTS_DIR}/{DATASET}_bootstrap_FD.xlsx"
with pd.ExcelWriter(out) as w:
    per_model.to_excel(w, sheet_name="per_model"); pairs.to_excel(w, sheet_name="pairs", index=False)
pd.set_option("display.width", 220)
print("\nper model (fd_point = sheet value; boot_* describe the spread, not a new FD):")
print(per_model.round(2).to_string())
print("\npaired differences A - B (negative = A better); P(A better) = share of replicates with FD_A < FD_B:")
print(pairs.round(3).to_string(index=False))
print(f"\nsaved {out}  ({time.time() - t0:.0f} s)")


## Guidance-tolerance curve (Step 24)

Scores the `*_n1000_*_curve_cfgXX.zip` sets (Drive folder `Sets_curve_n1000` next to `Sets_200k`) and the first 1,000 images of the existing 1.0 / 1.5 / 2.0 sets of the same three models, all at the same sample size, and draws FD, KD and failure rate against the guidance scale. Run after CONFIG, same session; re-run when more zips arrive.

In [ ]:
# ==== GUIDANCE-TOLERANCE CURVE: FD, KD and failure rate against the guidance scale, every point at 1,000 images ==========
# Works unchanged in the Cityscapes, Traffic and COCO Step14 notebooks (the dataset is read from XLSX). Three models: FDGAN,
# ADM* (mask-conditioned ADM) and FDGAN without box noise, guidance 1.0 .. 3.5, final sampler (ancestral 500, learned variance,
# colour guidance). Sources: the *_n1000_*_curve_cfgXX.zip files in CURVE_DIR (= Drive folder Sets_curve_n1000 next to Sets_200k)
# plus the existing 3,000-image sets of the same chain in SETS_DIR (…anc500_nocfg = 1.0, …anc500_cfg15, …anc500_cfg20), of which
# only the FIRST 1,000 images are used, so that every point has the same sample size (FD grows as the sample shrinks, so the
# 3,000-image values of the main sheet must never be mixed into this curve). Real side as in the main sheet. Writes
# <Dataset>_guidance_curve_n1000.xlsx and .png into RESULTS_DIR; points already in the xlsx are skipped, so the cell can be
# re-run when more zips arrive. Run after the CONFIG cell, same session. ~3 min per point.
N_CURVE = 1000
# ==========================================================================================================================
import os, re, glob, json, shutil, zipfile, subprocess, time
import numpy as np, pandas as pd
from PIL import Image
assert all(n in globals() for n in ("REAL_DIR", "TEST_DIR", "SETS_DIR", "RESULTS_DIR", "XLSX", "ENV", "parse", "image_stats")), \
    "run the cells above (including CONFIG) first"
DATASET = os.path.basename(XLSX).split("_200k")[0]                      # Cityscapes | Traffic | COCO
PREFIX, SUFFIX = {"Cityscapes": ("cityscapes_", ""), "Traffic": ("traffic2_", ""), "COCO": ("coco_", "_lsun")}[DATASET]
CURVE_DIR = os.path.join(os.path.dirname(os.path.normpath(SETS_DIR)), "Sets_curve_n1000")
OUT_XLSX = f"{RESULTS_DIR}/{DATASET}_guidance_curve_n1000.xlsx"
OUT_PNG  = f"{RESULTS_DIR}/{DATASET}_guidance_curve_n1000.png"
MODELS = {"fdgan_mask": "FDGAN", "adm_mask": "ADM* (mask-conditioned ADM)", "fdgan_nonoise": "FDGAN without box noise"}
PAT = re.compile(r"^" + PREFIX + r"(fdgan_mask|adm_mask|fdgan_nonoise)" + SUFFIX +
                 r"_ema200000_n(3000|1000)_r500_colsample_(?:anc500_|curve_)(nocfg|cfg(\d\d))$")
print(f"dataset {DATASET} | curve zips from {CURVE_DIR} | 3,000-image sets from {SETS_DIR}")

# ---- 1. collect the sources: (model, cfg) -> zip; a curve zip wins over a 3,000-image set of the same point -------------
sources = {}
for folder in (SETS_DIR, CURVE_DIR):
    for zp in sorted(glob.glob(f"{folder}/*.zip")):
        tag = os.path.basename(zp)[:-4]
        m = PAT.match(tag)
        if not m:
            continue
        model, n, lab, dd = m.group(1), int(m.group(2)), m.group(3), m.group(4)
        cfg = 1.0 if lab == "nocfg" else int(dd) / 10.0
        if (model, cfg) in sources and n == 3000:
            continue
        sources[(model, cfg)] = (zp, n)
print(f"{len(sources)} curve points found:")
for (model, cfg), (zp, n) in sorted(sources.items()):
    print(f"   {MODELS[model]:<34} CFG {cfg:.1f}   {os.path.basename(zp)}  ({n} images in zip)")
missing = [(MODELS[mo], c) for mo in MODELS for c in (1.0, 1.5, 2.0, 2.5, 3.0, 3.5) if (mo, c) not in sources]
if missing:
    print("not sampled yet (the figure will have a gap there):", missing)

# ---- 2. score each point on its first N_CURVE images --------------------------------------------------------------------
done = pd.read_excel(OUT_XLSX, index_col=0) if os.path.exists(OUT_XLSX) else pd.DataFrame()
done_src = set(done["source"].astype(str)) if "source" in done.columns else set()

def stage_first(zp, n_keep):
    tag = os.path.basename(zp)[:-4]
    raw, gen = f"/content/dgm-eval/_raw_{tag}", f"/content/dgm-eval/gen_curve_{tag}"
    shutil.rmtree(raw, ignore_errors=True); shutil.rmtree(gen, ignore_errors=True); os.makedirs(gen)
    with zipfile.ZipFile(zp) as z:
        z.extractall(raw)
    meta, pngs = {}, []
    for dp, dn, fn in os.walk(raw):
        for f in fn:
            if f == "meta.json":
                meta = json.load(open(os.path.join(dp, f)))
            elif f.startswith("sample_") and f.lower().endswith(".png"):
                pngs.append(os.path.join(dp, f))
    for p in sorted(pngs, key=os.path.basename)[:n_keep]:
        shutil.copy2(p, os.path.join(gen, os.path.basename(p)))
    shutil.rmtree(raw)
    bad = 0
    for f in sorted(os.listdir(gen)):
        try:
            with Image.open(os.path.join(gen, f)) as im:
                im.verify()
        except Exception:
            os.remove(os.path.join(gen, f)); bad += 1
    return gen, meta, bad

def dgm(model, metrics, gen, tag):
    txt = f"{RESULTS_DIR}/{tag}_curve_n{N_CURVE}_{model}.txt"
    cmd = ["python", "-m", "dgm_eval", REAL_DIR, gen, "--test_path", TEST_DIR, "--model", model, "--metrics", *metrics.split(),
           "--device", "cuda", "--batch_size", "128", "--nsample", "3000", "--seed", "42",
           "--output_dir", f"/content/dgm-eval/experiments_{DATASET.lower()}_curve"]
    with open(txt, "w") as f:
        p = subprocess.run(cmd, cwd="/content/dgm-eval", env=ENV, stdout=f, stderr=subprocess.STDOUT)
    res = parse(txt)
    if p.returncode != 0 or "fd" not in res:
        print(open(txt).read()[-3000:]); raise RuntimeError(f"dgm-eval failed for {tag} / {model}: see {txt}")
    return res

for (model, cfg), (zp, n) in sorted(sources.items()):
    tag = os.path.basename(zp)[:-4]
    if tag in done_src:
        print("skip (done):", tag); continue
    t0 = time.time(); print(f"\n===== {MODELS[model]}  CFG {cfg:.1f}  <- {tag}")
    gen, meta, bad = stage_first(zp, N_CURVE)
    stats = image_stats(gen)
    print(f"staged {stats['n_images']} images ({bad} corrupt removed) | saturation {stats['saturation']:.1f} contrast {stats['contrast']:.1f} "
          f"failure rate {stats['failure_rate_pct']:.2f}%")
    A = dgm("dinov2", "fd kd prdc", gen, tag)
    B = dgm("inception", "fd", gen, tag)
    row = {"model": MODELS[model], "model_key": model, "cfg": cfg, "source": tag, "n_images": stats["n_images"],
           "dinov2_fd": A.get("fd"), "dinov2_kd_value": A.get("kd_value"), "dinov2_precision": A.get("precision"), "dinov2_recall": A.get("recall"),
           "inception_fd": B.get("fd"), "failure_rate_pct": stats["failure_rate_pct"], "grain_median": stats["grain_median"],
           "saturation": stats["saturation"], "contrast": stats["contrast"], "brightness": stats["brightness"],
           "unet_evals_per_image": meta.get("cost", {}).get("unet_evals_per_image"), "mask_cfg_scale_meta": meta.get("args", {}).get("mask_cfg_scale")}
    done = pd.concat([done, pd.DataFrame([row], index=[f"{model}_cfg{cfg:.1f}"])])
    done.index.name = "point"; done.to_excel(OUT_XLSX)
    print(f"DINOv2 FD {A['fd']:.1f} | KD {A.get('kd_value', float('nan')):.3f} | FID {B['fd']:.2f} | P {A.get('precision', float('nan')):.3f} "
          f"R {A.get('recall', float('nan')):.3f} | {time.time() - t0:.0f} s -> saved")
    shutil.rmtree(gen, ignore_errors=True)

# ---- 3. table and figure ---------------------------------------------------------------------------------------------------
tab = pd.read_excel(OUT_XLSX, index_col=0).sort_values(["model_key", "cfg"])
pd.set_option("display.width", 220)
print("\n" + tab[["model", "cfg", "n_images", "dinov2_fd", "dinov2_kd_value", "inception_fd", "dinov2_precision", "dinov2_recall",
                   "failure_rate_pct", "saturation", "contrast"]].round(3).to_string(index=False))
if DATASET == "COCO":
    print("note: the failure-rate threshold (Laplacian variance > 2000) is not valid on COCO (25.6 % of REAL potted-plant images exceed it); "
          "read grain_median against the real value 1,339 instead")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for key, name in MODELS.items():
    t = tab[tab["model_key"] == key].sort_values("cfg")
    if len(t) == 0:
        continue
    ax[0].plot(t["cfg"], t["dinov2_fd"], marker="o", label=name)
    ax[1].plot(t["cfg"], t["dinov2_kd_value"], marker="o", label=name)
    ax[2].plot(t["cfg"], t["failure_rate_pct"], marker="o", label=name)
for a, yl in zip(ax, ["DINOv2 FD (1,000 images)", "DINOv2 KD", "failure rate (%)"]):
    a.set_xlabel("mask guidance scale"); a.set_ylabel(yl); a.grid(alpha=0.3)
ax[0].legend(fontsize=8)
fig.suptitle(f"{DATASET}, final sampler: tolerance to guidance strength")
fig.tight_layout(); fig.savefig(OUT_PNG, dpi=150); plt.show()
print("saved", OUT_XLSX, "and", OUT_PNG)
